In [1]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW # 从 torch 导入避开 transformers 报错
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score
# --- 1. M4 Pro 核心加速配置 ---
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("检测到 M4 Pro 芯片，已启用 MPS 加速！")
else:
    device = torch.device("cpu")
    print("未检测到硬件加速，使用 CPU。")

MODEL_NAME = 'distilbert-base-uncased'
BATCH_SIZE = 32  # M4 Pro 内存大，可以尝试 32 甚至 64
MAX_LEN = 256

# --- 2. 数据加载 ---
train_df = pd.concat([pd.read_csv('train.csv'), pd.read_csv('val.csv')])
test_df = pd.read_csv('test.csv')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN)
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_loader = DataLoader(TextDataset(train_df['text'].astype(str).tolist(), train_df['label'].tolist()), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TextDataset(test_df['text'].astype(str).tolist(), test_df['label'].tolist()), batch_size=BATCH_SIZE)

# --- 3. 构建模型 ---
class BertClassifier(torch.nn.Module):
    def __init__(self):
        super(BertClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        # 初始冻结：先看看 BERT 原本提取特征的能力
        for param in self.bert.parameters():
            param.requires_grad = False
        self.classifier = torch.nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # 提取 [CLS] 向量
        return self.classifier(outputs.last_hidden_state[:, 0, :])

model = BertClassifier().to(device)
# 针对分类头设置稍大的学习率
optimizer = AdamW(model.classifier.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

# --- 4. 训练循环 ---
print("开始训练...")
for epoch in range(3): # 跑 3 个 epoch 看看
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader):
        optimizer.zero_grad()
        # 将数据搬运到 MPS
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Avg Loss: {total_loss/len(train_loader):.4f}")

# --- 5. 最终测试 ---
model.eval()
all_preds = []
all_labels = []

print("\n正在生成评估报告...")
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)
        
        # 将结果搬回 CPU 并转为 list
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 生成最终的视觉化报告
acc = accuracy_score(all_labels, all_preds)
report = classification_report(all_labels, all_preds, digits=4)

print("\n" + "="*40)
print("最终优化方案评估报告:")
print(f"准确率: {acc:.4f}")
print("-" * 40)
print(report)
print("="*40)

ModuleNotFoundError: No module named 'torch'